# Experiment: Recipient Source Privacy Gate

Objective: classify whether a public recipient evidence card preserves source auditability without publishing raw candidate URLs, real identities, contacts, private replies, patient material, treatment, trial, purchase, import, or procurement scope.


In [ ]:
from __future__ import annotations

READY = "recipient_source_privacy_public_card_ready"
MISSING_INDEX = "recipient_source_privacy_hold_missing_private_index"
SOURCE_FREE = "recipient_source_privacy_hold_source_free"
BLOCKED = "recipient_source_privacy_release_blocked_identifying_source"

REQUIRED = {
    "source_bundle_ref",
    "source_kind_labels",
    "source_count_label",
    "model_capability",
    "endpoint_capability",
    "raw_data_export",
    "cost_timeline_constraints",
    "ethics_biosafety_constraints",
}
IDENTIFYING_BLOCKERS = {
    "raw_candidate_url",
    "real_recipient_name",
    "person_name",
    "email_or_phone",
    "address",
    "private_reply",
    "patient_record",
    "patient_sample",
    "treatment_or_trial_scope",
    "purchase_import_procurement",
}


def classify_source_privacy(card: dict[str, bool | str]) -> str:
    """Return the public-safe source-privacy label for a card."""
    if any(card.get(flag, False) for flag in IDENTIFYING_BLOCKERS):
        return BLOCKED
    if card.get("source_free_claim", False):
        return SOURCE_FREE
    if any(not card.get(flag, False) for flag in REQUIRED):
        return MISSING_INDEX
    return READY


base_card = {flag: True for flag in REQUIRED}
base_card["recipient_alias"] = "recipient-candidate-001"
base_card["source_bundle_ref"] = "private-source-bundle-001"

cases = {
    "ready": base_card,
    "missing_bundle": {**base_card, "source_bundle_ref": False},
    "source_free": {**base_card, "source_free_claim": True},
    "raw_url": {**base_card, "raw_candidate_url": True},
    "private_reply": {**base_card, "private_reply": True},
    "patient_sample": {**base_card, "patient_sample": True},
}
expected = {
    "ready": READY,
    "missing_bundle": MISSING_INDEX,
    "source_free": SOURCE_FREE,
    "raw_url": BLOCKED,
    "private_reply": BLOCKED,
    "patient_sample": BLOCKED,
}
results = {name: classify_source_privacy(card) for name, card in cases.items()}
assert results == expected

decision_summary = {
    "current_state": "private_index_required_before_public_card",
    "ready_label": READY,
    "blocked_label": BLOCKED,
    "boundary": "raw candidate URLs stay private; public cards use bundle refs",
}
decision_summary

## Decision

A real recipient candidate cannot enter the public repo as raw URLs or names. Public cards should carry a private source-bundle reference and only de-identified capability labels.
